# Capítulo 4 — Escoamento em Tubulações e Bombeamento

**Autor:** Jader Lugon Junior  
**Livro:** Fenômenos de Transporte: Fundamentos e Modelagem Computacional  
**Repositório:** [github.com/JaderLugon/fenomenos-transporte-livro](https://github.com/JaderLugon/fenomenos-transporte-livro)

---

## 🎯 Objetivos de Aprendizagem

Ao final deste notebook, você será capaz de:

1. **Aplicar** a equação de Bernoulli estendida com perdas de carga
2. **Calcular** perda de carga distribuída pela equação de Darcy-Weisbach
3. **Determinar** o fator de atrito pelas correlações de Colebrook-White e Swamee-Jain
4. **Calcular** perdas de carga localizadas em acessórios
5. **Diferenciar** escoamento laminar (Hagen-Poiseuille) e turbulento (Lei de 1/7)
6. **Dimensionar** bombas e verificar NPSH para evitar cavitação

---

## 📚 Conteúdo Teórico Resumido

### 4.1 Equação de Bernoulli Estendida

Para escoamento real (com perdas e bombas):

$$\frac{p_1}{\gamma} + \frac{V_1^2}{2g} + z_1 + H_B = \frac{p_2}{\gamma} + \frac{V_2^2}{2g} + z_2 + h_f + h_s$$

onde:
- $H_B$ = altura manométrica da bomba [m]
- $h_f$ = perda de carga distribuída [m]
- $h_s$ = perda de carga localizada (singular) [m]

### 4.2 Perda de Carga Distribuída (Darcy-Weisbach)

$$h_f = f \cdot \frac{L}{D} \cdot \frac{V^2}{2g}$$

onde $f$ é o fator de atrito, que depende de $\mathrm{Re}$ e $\varepsilon/D$.

### 4.3 Fator de Atrito

| Regime | Fórmula |
|--------|---------|
| Laminar ($\mathrm{Re} < 2000$) | $f = 64/\mathrm{Re}$ |
| Turbulento (Colebrook-White) | $\frac{1}{\sqrt{f}} = -2\log_{10}\left(\frac{\varepsilon/D}{3,7} + \frac{2,51}{\mathrm{Re}\sqrt{f}}\right)$ |
| Turbulento (Swamee-Jain) | $f = \frac{0,25}{\left[\log_{10}\left(\frac{\varepsilon/D}{3,7} + \frac{5,74}{\mathrm{Re}^{0,9}}\right)\right]^2}$ |

### 4.4 Perdas Localizadas

$$h_s = K \cdot \frac{V^2}{2g}$$

onde $K$ é o coeficiente de perda do acessório (válvula, curva, etc.).

### 4.5 Escoamento Laminar (Hagen-Poiseuille)

Perfil parabólico:
$$u(r) = u_{\max}\left(1 - \frac{r^2}{R^2}\right)$$

Vazão:
$$Q = \frac{\pi R^4 \Delta p}{8\mu L}$$

### 4.6 Escoamento Turbulento (Lei de 1/7)

Perfil aproximado:
$$\frac{u}{u_{\max}} = \left(1 - \frac{r}{R}\right)^{1/7}$$

Velocidade média:
$$U \approx 0,817 \cdot u_{\max}$$

### 4.7 Potência de Bomba

$$P = \frac{\gamma \cdot Q \cdot H_B}{\eta}$$

onde $\eta$ é o rendimento da bomba.

---

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import sys
import os

# Adiciona o diretório raiz ao path para importar módulos
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from src import hidrodinamica, utils

# Configuração de plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("✅ Ambiente configurado com sucesso!")
print(f"📦 NumPy {np.__version__} | Matplotlib {plt.matplotlib.__version__}")

---

## 🔬 Seção 1: Perda de Carga Distribuída

### Exercício 1.1: Perda de Carga em Tubo de Ferro Galvanizado

Calcule a perda de carga em um tubo de ferro galvanizado ($\varepsilon = 0,15$ mm) com $D = 75$ mm, $L = 80$ m, transportando água ($\nu = 10^{-6}$ m²/s) com vazão $Q = 3,0$ L/s.

In [ ]:
# ============================================================
# EXERCÍCIO 1.1: Perda de Carga Distribuída
# ============================================================

print("=" * 60)
print("EXERCÍCIO 1.1: PERDA DE CARGA EM TUBO DE FERRO GALVANIZADO")
print("=" * 60)

# Dados do problema
D = 0.075        # m (diâmetro)
L = 80.0         # m (comprimento)
Q = 3.0e-3       # m³/s (vazão)
nu = 1.0e-6      # m²/s (viscosidade cinemática da água)
eps = 0.15e-3    # m (rugosidade do ferro galvanizado)
g = 9.81         # m/s²

# Cálculos
A = np.pi * D**2 / 4
V = Q / A
Re = V * D / nu
eps_D = eps / D

print(f"\n📊 Dados do problema:")
print(f"  • Diâmetro: D = {D*1000:.0f} mm")
print(f"  • Comprimento: L = {L} m")
print(f"  • Vazão: Q = {Q*1000:.1f} L/s")
print(f"  • Rugosidade: ε = {eps*1000:.2f} mm (ferro galvanizado)")

print(f"\n🧮 Cálculos intermediários:")
print(f"  • Área: A = π·D²/4 = {A:.4f} m²")
print(f"  • Velocidade: V = Q/A = {V:.3f} m/s")
print(f"  • Reynolds: Re = V·D/ν = {Re:.0f}")
print(f"  • Rugosidade relativa: ε/D = {eps_D:.4f}")

# Determinar regime e fator de atrito
if Re < 2000:
    f = 64 / Re
    regime = "LAMINAR"
else:
    f = hidrodinamica.fator_atrito_swamee_jain(Re, eps_D)
    regime = "TURBULENTO"

print(f"\n🔍 Regime: {regime}")
print(f"  • Fator de atrito: f = {f:.4f}")

# Perda de carga
hf = hidrodinamica.perda_carga_darcy(f, L, D, V, g)

print(f"\n📊 Resultado:")
print(f"  • Perda de carga: h_f = f·(L/D)·(V²/2g) = {hf:.3f} m")
print(f"  • Perda por metro: h_f/L = {hf/L:.4f} m/m = {hf/L*1000:.2f} ‰")

print(f"\n💡 Interpretação:")
print(f"  A perda de {hf:.2f} m em {L} m de tubo ({hf/L*100:.2f}%) é moderada.")
print(f"  Para uma tubulação de ferro galvanizado, a rugosidade relativa")
print(f"  (ε/D = {eps_D:.4f}) coloca o escoamento na zona de transição do Diagrama de Moody.")

---

## 🧪 Seção 2: Fator de Atrito (Colebrook-White vs. Swamee-Jain)

### Exercício 2.1: Comparação entre Correlações

Compare o fator de atrito calculado por Colebrook-White (iterativo) e Swamee-Jain (explícito) para diferentes valores de Reynolds.

In [ ]:
# ============================================================
# EXERCÍCIO 2.1: Comparação de Correlações
# ============================================================

print("=" * 60)
print("EXERCÍCIO 2.1: COMPARAÇÃO COLEBROOK-WHITE vs. SWAMEE-JAIN")
print("=" * 60)

# Parâmetros
eps_D = 1e-4  # rugosidade relativa
Re_values = np.logspace(4, 7, 10)  # Reynolds de 10^4 a 10^7

print(f"\n📊 Rugosidade relativa: ε/D = {eps_D:.1e}")
print(f"\n{'Re':<12} {'f (Colebrook)':<15} {'f (Swamee-Jain)':<15} {'Erro (%)':<10}")
print("-" * 55)

for Re in Re_values:
    f_colebrook = hidrodinamica.fator_atrito_colebrook(Re, eps_D)
    f_swamee = hidrodinamica.fator_atrito_swamee_jain(Re, eps_D)
    erro = abs(f_colebrook - f_swamee) / f_colebrook * 100
    print(f"{Re:<12.0e} {f_colebrook:<15.5f} {f_swamee:<15.5f} {erro:<10.2f}")

print(f"\n💡 Interpretação:")
print(f"  A correlação de Swamee-Jain é precisa dentro de ~1% para a maioria")
print(f"  das aplicações de engenharia, mas é muito mais rápida computacionalmente")
print(f"  por ser explícita (não requer iteração).")

---

## 🔧 Seção 3: Perdas Localizadas

### Exercício 3.1: Sistema com Acessórios

Calcule a perda de carga localizada total em um sistema com: 2 válvulas de gaveta ($K = 0,15$), 6 curvas 90° ($K = 0,9$) e 1 válvula de retenção ($K = 2,5$), com $V = 2,0$ m/s.

In [ ]:
# ============================================================
# EXERCÍCIO 3.1: Perdas Localizadas
# ============================================================

print("=" * 60)
print("EXERCÍCIO 3.1: PERDA DE CARGA LOCALIZADA EM SISTEMA")
print("=" * 60)

# Dados
acessorios = {
    "Válvula de gaveta": {"K": 0.15, "quantidade": 2},
    "Curva 90°": {"K": 0.9, "quantidade": 6},
    "Válvula de retenção": {"K": 2.5, "quantidade": 1}
}
V = 2.0  # m/s
g = 9.81  # m/s²

print(f"\n📊 Acessórios do sistema:")
print(f"{'Acessório':<25} {'K':<8} {'Qtd':<6} {'K_total':<10}")
print("-" * 50)

K_total = 0
for nome, dados in acessorios.items():
    K_item = dados["K"] * dados["quantidade"]
    K_total += K_item
    print(f"{nome:<25} {dados['K']:<8.2f} {dados['quantidade']:<6} {K_item:<10.2f}")

print("-" * 50)
print(f"{'TOTAL':<25} {'':<8} {'':<6} {K_total:<10.2f}")

# Perda de carga localizada
hs = hidrodinamica.perda_carga_localizada(K_total, V, g)

print(f"\n🧮 Cálculo:")
print(f"  • h_s = K_total · V²/(2g) = {K_total:.2f} × {V}²/(2×{g})")
print(f"  • h_s = {hs:.3f} m")

print(f"\n💡 Interpretação:")
print(f"  As curvas 90° contribuem com {6*0.9/K_total*100:.1f}% da perda total.")
print(f"  Em projetos otimizados, pode-se substituir curvas de 90° por curvas")
print(f"  de 45° ou curvas de raio longo (K = 0,6) para reduzir a perda.")

---

## 🌊 Seção 4: Escoamento Laminar (Hagen-Poiseuille)

### Exercício 4.1: Perfil de Velocidade e Vazão

Um óleo ($\mu = 0,1$ Pa·s, $\rho = 850$ kg/m³) escoa em um tubo horizontal de $D = 5$ cm com velocidade máxima $u_{\max} = 0,8$ m/s. Determine: (a) velocidade média, (b) vazão, (c) perda de carga em $L = 20$ m, (d) tensão na parede.

In [ ]:
# ============================================================
# EXERCÍCIO 4.1: Escoamento Laminar (Hagen-Poiseuille)
# ============================================================

print("=" * 60)
print("EXERCÍCIO 4.1: ESCOAMENTO LAMINAR - HAGEN-POISEUILLE")
print("=" * 60)

# Dados
mu = 0.1         # Pa·s (viscosidade dinâmica)
rho = 850        # kg/m³ (densidade)
D = 0.05         # m (diâmetro)
R = D / 2        # m (raio)
u_max = 0.8      # m/s (velocidade máxima)
L = 20.0         # m (comprimento)
g = 9.81         # m/s²

# (a) Velocidade média (laminar: U = u_max / 2)
U = u_max / 2

# Verificar Reynolds
Re = U * D * rho / mu

print(f"\n📊 Dados do problema:")
print(f"  • Viscosidade dinâmica: μ = {mu} Pa·s")
print(f"  • Densidade: ρ = {rho} kg/m³")
print(f"  • Diâmetro: D = {D*1000:.0f} mm")
print(f"  • Velocidade máxima: u_max = {u_max} m/s")
print(f"  • Comprimento: L = {L} m")

print(f"\n🧮 (a) Velocidade média:")
print(f"  • U = u_max / 2 = {u_max} / 2 = {U} m/s")
print(f"  • Reynolds: Re = U·D·ρ/μ = {Re:.0f}")
print(f"  • Regime: {'LAMINAR' if Re < 2000 else 'TURBULENTO'}")

# (b) Vazão
A = np.pi * D**2 / 4
Q = U * A

print(f"\n🧮 (b) Vazão volumétrica:")
print(f"  • Área: A = π·D²/4 = {A:.5f} m²")
print(f"  • Q = U·A = {U} × {A:.5f} = {Q:.5f} m³/s = {Q*1000:.3f} L/s")

# (c) Perda de carga (Hagen-Poiseuille)
hf = 32 * mu * L * U / (rho * g * D**2)

print(f"\n🧮 (c) Perda de carga (Hagen-Poiseuille):")
print(f"  • h_f = 32·μ·L·U / (ρ·g·D²)")
print(f"  • h_f = 32 × {mu} × {L} × {U} / ({rho} × {g} × {D}²)")
print(f"  • h_f = {hf:.4f} m")

# (d) Tensão na parede
tau_w = 4 * mu * U / R

print(f"\n🧮 (d) Tensão de cisalhamento na parede:")
print(f"  • τ_w = 4·μ·U / R = 4 × {mu} × {U} / {R}")
print(f"  • τ_w = {tau_w:.2f} Pa")

print(f"\n💡 Interpretação:")
print(f"  A relação U = u_max/2 é exclusiva do regime laminar.")
print(f"  Em regime turbulento, essa razão é tipicamente 0,8-0,85 (perfil mais achatado).")

---

## 🌪️ Seção 5: Escoamento Turbulento (Lei de 1/7)

### Exercício 5.1: Comparação de Perfis

Para um tubo de $D = 10$ cm com $u_{\max} = 2$ m/s em regime turbulento, compare a vazão e a velocidade média com o caso laminar.

In [ ]:
# ============================================================
# EXERCÍCIO 5.1: Comparação Laminar vs. Turbulento
# ============================================================

print("=" * 60)
print("EXERCÍCIO 5.1: COMPARAÇÃO LAMINAR vs. TURBULENTO")
print("=" * 60)

# Dados
D = 0.10       # m
R = D / 2      # m
u_max = 2.0    # m/s

# Turbulento (Lei de 1/7)
U_turb = 49/60 * u_max
Q_turb = U_turb * np.pi * D**2 / 4

# Laminar (parabólico)
U_lam = u_max / 2
Q_lam = U_lam * np.pi * D**2 / 4

print(f"\n📊 Dados do problema:")
print(f"  • Diâmetro: D = {D*1000:.0f} mm")
print(f"  • Velocidade máxima: u_max = {u_max} m/s")

print(f"\n🔍 Resultados:")
print(f"\n  {'Parâmetro':<25} {'Laminar':<15} {'Turbulento':<15} {'Razão':<10}")
print("  " + "-" * 65)
print(f"  {'U [m/s]':<25} {U_lam:<15.3f} {U_turb:<15.3f} {U_turb/U_lam:<10.2f}")
print(f"  {'Q [L/s]':<25} {Q_lam*1000:<15.3f} {Q_turb*1000:<15.3f} {Q_turb/Q_lam:<10.2f}")

print(f"\n💡 Interpretação:")
print(f"  O regime turbulento transporta {(Q_turb/Q_lam - 1)*100:.1f}% mais vazão")
print(f"  para a mesma velocidade máxima, devido ao perfil mais 'achatado'.")
print(f"  A razão U/u_max é 0,50 (laminar) vs. 0,817 (turbulento).")

# Visualização dos perfis
r = np.linspace(0, R, 100)

# Perfil laminar
u_lam = u_max * (1 - (r/R)**2)

# Perfil turbulento (Lei de 1/7)
u_turb = u_max * (1 - r/R)**(1/7)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(u_lam, r*100, 'b-', linewidth=2, label='Laminar (parabólico)')
ax.plot(u_turb, r*100, 'r-', linewidth=2, label='Turbulento (Lei de 1/7)')
ax.set_xlabel('Velocidade, u [m/s]', fontsize=12)
ax.set_ylabel('Posição radial, r [cm]', fontsize=12)
ax.set_title('Perfis de Velocidade em Tubo Circular', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 🔋 Seção 6: Dimensionamento de Bombas

### Exercício 6.1: Potência e Custo Energético

Determine a potência elétrica necessária para uma bomba que deve fornecer $H_B = 30$ m com vazão $Q = 5$ L/s, considerando rendimento $\eta = 0,70$. Qual seria o custo mensal (30 dias) se a bomba operar 6 h/dia e a tarifa for R\$ 0,90/kWh?

In [ ]:
# ============================================================
# EXERCÍCIO 6.1: Dimensionamento de Bomba
# ============================================================

print("=" * 60)
print("EXERCÍCIO 6.1: DIMENSIONAMENTO DE BOMBA")
print("=" * 60)

# Dados
HB = 30.0        # m (altura manométrica)
Q = 5.0e-3       # m³/s (vazão)
eta = 0.70       # rendimento da bomba
gamma = 9790     # N/m³ (peso específico da água)
tarifa = 0.90    # R$/kWh
horas_dia = 6    # h/dia
dias = 30        # dias

# Cálculos
P_hid = gamma * Q * HB
P_el = P_hid / eta
energia_mes = P_el/1000 * horas_dia * dias  # kWh
custo_mes = energia_mes * tarifa

print(f"\n📊 Dados do problema:")
print(f"  • Altura manométrica: H_B = {HB} m")
print(f"  • Vazão: Q = {Q*1000:.1f} L/s")
print(f"  • Rendimento: η = {eta}")
print(f"  • Tarifa: R$ {tarifa:.2f}/kWh")
print(f"  • Operação: {horas_dia} h/dia × {dias} dias")

print(f"\n🧮 Cálculos:")
print(f"  • Potência hidráulica: P_hid = γ·Q·H_B = {gamma} × {Q*1000:.1f}×10⁻³ × {HB}")
print(f"                       = {P_hid:.1f} W = {P_hid/1000:.2f} kW")
print(f"  • Potência elétrica: P_el = P_hid/η = {P_hid/1000:.2f}/{eta}")
print(f"                     = {P_el/1000:.2f} kW")
print(f"  • Energia mensal: E = {P_el/1000:.2f} × {horas_dia} × {dias}")
print(f"                  = {energia_mes:.1f} kWh")
print(f"  • Custo mensal: C = {energia_mes:.1f} × {tarifa:.2f}")
print(f"                = R$ {custo_mes:.2f}")

print(f"\n💡 Interpretação:")
print(f"  O rendimento da bomba (η = 0,70) significa que 30% da energia")
print(f"  elétrica é dissipada em perdas mecânicas e hidráulicas.")
print(f"  Bombas de alta eficiência (η > 0,85) reduziriam o custo operacional.")

---

## 🔬 Estudos de Caso

Os estudos de caso aplicam os conceitos deste capítulo em problemas reais de engenharia.
Clique nos links abaixo para abrir os notebooks interativos:

| Estudo de Caso | Descrição | Link |
|----------------|-----------|------|
| **Caso 4.1** | Solver Colebrook-White | [🔗 Abrir](../casos/04_1_Solver_Colebrook_White.ipynb) |
| **Caso 4.2** | Bombeamento em Prédio Residencial | [🔗 Abrir](../casos/04_2_Bombeamento_Predio_Residencial.ipynb) |
| **Caso 4.3** | Sistema Hidráulico para Carga Pesada | [🔗 Abrir](../casos/04_3_Sistema_Hidraulico_Carga_Pesada.ipynb) |

> 💡 **Dica:** Os estudos de caso podem ser executados independentemente deste notebook principal.

---

## 📖 Referências

- MUNSON, B. R. et al. *Fundamentos da Mecânica dos Fluidos*. 6ª ed. Blucher, 2010.
- WHITE, F. M. *Fluid Mechanics*. 7ª ed. McGraw-Hill, 2011.
- MACINTYRE, A. J. *Instalações Hidráulicas*. 2ª ed. LTC, 1996.
- BIRD, R. B. et al. *Transport Phenomena*. 2ª ed. Wiley, 2002.

---

## 🔄 Navegação

- [📚 Capítulo Anterior: Modelagem Matemática](03_balancos_conservacao.ipynb)
- [📚 Próximo Capítulo: Hidrodinâmica de Canais](05_hidrodinamica_canais_abertos.ipynb)
- [📂 Outros Capítulos](../notebooks/)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**QR Code do Capítulo 4**  
Aponte seu celular para o QR Code no livro para acessar este notebook!

</div>

In [ ]:
print("=" * 60)
print("✅ NOTEBOOK CONCLUÍDO COM SUCESSO!")
print("=" * 60)
print("\n🎓 Você completou o Capítulo 4!")
print("📖 Próximo passo: Capítulo 5 - Hidrodinâmica de Canais Abertos")
print("\n💡 Dica: Execute este notebook quantas vezes precisar!")
print("   Modifique os parâmetros e explore diferentes cenários.")